#### Decision API 2차 배치 검수

`ner_human_review.review_method = 'LLM검수'` 대상만 PostgreSQL에서 스트리밍 추출하고, 언론사별로 태그 오류와 태깅 병합 필요 여부를 판단한다.

- API 실행은 3번 셀에서 `CURRENT_PRESS`로 지정한 언론사 1곳씩 진행한다.
- 안전 기준 초과 또는 10건 테스트 실패 시 전체 실행은 자동 중단한다.
- 완료된 식별자는 언론사별 `results.jsonl`에서 읽어 재호출하지 않는다.

In [2]:
import os
import sys
from datetime import datetime
from itertools import islice
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path(r'C:\work\ner')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import get_postgre_connect
from src.ner_review.decision.decision import DECISION_MODEL
from src.ner_review.decision.batch import (
    DEFAULT_BATCH_SIZE,
    DEFAULT_MAX_WORKERS,
    DEFAULT_MAX_RETRIES,
    DEFAULT_REQUEST_TIMEOUT_SECONDS,
    build_decision_input,
    estimate_usage,
    run_press_list,
    stream_review_targets,
)

DATA_VERSION = None # os.getenv('DATA_VERSION') or None
MODEL_OVERRIDE = None  # Decisions API 공식 지원 모델 외 값은 사용하지 않음
MODEL = MODEL_OVERRIDE or DECISION_MODEL
if MODEL != 'gpt-6-luna':
    raise ValueError(f'Decisions API 지원 모델은 gpt-6-luna입니다: {MODEL}')
RUN_API = True  # 3번 셀에서 CURRENT_PRESS 언론사 1곳을 실행
PRESS_FILTER = None  # 예: ['강원일보']
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'decision_2' / f'run_{datetime.now():%Y%m%d_%H%M%S}'

# API 제한과 비용에 맞춰 조정하는 설정
BATCH_SIZE = DEFAULT_BATCH_SIZE
MAX_WORKERS = DEFAULT_MAX_WORKERS
MAX_RETRIES = DEFAULT_MAX_RETRIES
BACKOFF_SECONDS = 2.0
MAX_BACKOFF_SECONDS = 60.0
REQUEST_TIMEOUT_SECONDS = DEFAULT_REQUEST_TIMEOUT_SECONDS
MAX_INPUT_CHARS = 120_000
SAMPLE_SIZE = 100
CHARS_PER_TOKEN = 4.0
ESTIMATED_OUTPUT_TOKENS_PER_REQUEST = 300
INPUT_PRICE_PER_1M = None   # 모델 가격표 확인 후 숫자로 입력
OUTPUT_PRICE_PER_1M = None  # 모델 가격표 확인 후 숫자로 입력

print('output_root:', OUTPUT_ROOT)
print('model:', MODEL, '| run_api:', RUN_API)

output_root: C:\work\ner\outputs\decision_2\run_20261120_014739
model: gpt-6-luna | run_api: True


##### 1. 대상 언론사와 소규모 입력 확인

In [3]:
def list_press_names(data_version=None):
    where = "WHERE review_method = 'LLM검수'"
    params = []
    if data_version:
        where += ' AND data_version = %s'
        params.append(data_version)
    with get_postgre_connect() as connection:
        rows = connection.execute(
            f'SELECT DISTINCT press FROM ner_human_review {where} ORDER BY press',
            params,
        ).fetchall()
    return [row[0] for row in rows]

press_list = list_press_names(DATA_VERSION)
if PRESS_FILTER:
    press_list = [press for press in press_list if press in PRESS_FILTER]
print('presses:', press_list)

preview_rows = []
for press in press_list[:3]:
    target = next(stream_review_targets(data_version=DATA_VERSION, press=press), None)
    if target:
        preview_rows.append({
            'press': press,
            'data_version': target['data_version'],
            'doc_id': target['doc_id'],
            'sentence_id': target['sentence_id'],
            'original_form': target['original_form'],
            'entity_count': len(target['entities']),
            'active_rule_count': len(target['sentence_rule_evidence']),
        })
display(pd.DataFrame(preview_rows))

presses: ['강원일보', '경기일보', '경향신문', '노컷뉴스', '뉴스1', '뉴스핌', '뉴시스', '디지털타임스', '마이데일리', '부산일보', '서울신문', '세계일보', '스포츠경향', '스포츠투데이', '아시아투데이', '이데일리', '이투데이', '일간투데이', '충청일보']


,press,data_version,doc_id,sentence_id,original_form,entity_count,active_rule_count
0,강원일보,20261117_192958_강원일보,NLRW2600000001.10006,NLRW2600000001.10006.6.2,<2022년: DT> <20대 대선: EV>의 <강원: LC>지역 <1일차: DT>...,1,1
1,경기일보,20261117_193354_경기일보,NLRW2600000002.10002,NLRW2600000002.10002.1.1,"<박남춘: PS> “<이재명: PS> 원팀, 정권교체”…<1년: DT>만 SNS 글...",2,2
2,경향신문,20261117_193858_경향신문,NWRW2600000001.10004,NWRW2600000001.10004.5.1,앞서 <법사위: OG>는 <윤석열: PS> 대통령의 구속취소 결정에 대한 <이날: ...,1,1


##### 2. 언론사별 요청 수·토큰·비용 추정

전체 입력은 메모리에 모으지 않고, 언론사별 generator를 한 번 순회한다. 토큰은 문자 수 기반 보수적 추정이며 실제 사용량은 결과 JSONL의 `api.usage`를 확인한다.

In [3]:
usage_rows = []
for press in press_list:
    target_iter = stream_review_targets(data_version=DATA_VERSION, press=press)
    sample_targets = list(islice(target_iter, SAMPLE_SIZE))
    usage = estimate_usage(sample_targets, sample_size=SAMPLE_SIZE, chars_per_token=CHARS_PER_TOKEN)
    remaining_count = sum(1 for _ in target_iter)
    estimated_request_count = len(sample_targets) + remaining_count
    estimated_input_tokens = usage['estimated_input_tokens']
    estimated_input_tokens_total = estimated_request_count * estimated_input_tokens
    estimated_output_tokens_total = estimated_request_count * ESTIMATED_OUTPUT_TOKENS_PER_REQUEST
    estimated_cost = None
    if INPUT_PRICE_PER_1M is not None and OUTPUT_PRICE_PER_1M is not None:
        estimated_cost = (estimated_input_tokens_total / 1_000_000 * INPUT_PRICE_PER_1M) + (estimated_output_tokens_total / 1_000_000 * OUTPUT_PRICE_PER_1M)
    usage_rows.append({
        'press': press,
        'sample_size': usage['sample_size'],
        'estimated_request_count': estimated_request_count,
        'avg_input_chars': usage['avg_input_chars'],
        'max_input_chars': usage.get('max_input_chars', 0),
        'estimated_input_tokens_per_request': estimated_input_tokens,
        'estimated_input_tokens_total': estimated_input_tokens_total,
        'estimated_output_tokens_total': estimated_output_tokens_total,
        'estimated_cost': estimated_cost,
    })
usage_frame = pd.DataFrame(usage_rows)
display(usage_frame)
print('주의: 정확한 요청 수/사용 토큰은 각 press/summary.json에 누적 기록됩니다.')

,press,sample_size,estimated_request_count,avg_input_chars,max_input_chars,estimated_input_tokens_per_request,estimated_input_tokens_total,estimated_output_tokens_total,estimated_cost
0,강원일보,100,2177,504.8,765,126,274302,653100,None
1,경기일보,100,2706,537.7,863,134,362604,811800,None
2,경향신문,100,6372,525.1,797,131,834732,1911600,None
3,노컷뉴스,100,9076,551.5,1183,138,1252488,2722800,None
4,뉴스1,100,28855,545.4,840,136,3924280,8656500,None
5,뉴스핌,100,14403,552.1,1024,138,1987614,4320900,None
6,뉴시스,100,22106,534.3,1207,134,2962204,6631800,None
7,디지털타임스,100,5404,540.0,751,135,729540,1621200,None
8,마이데일리,100,7706,519.0,745,130,1001780,2311800,None
9,부산일보,100,6347,543.5,952,136,863192,1904100,None


주의: 정확한 요청 수/사용 토큰은 각 press/summary.json에 누적 기록됩니다.


##### 3. 언론사별 순차 배치 실행

`MAX_WORKERS`만 한 언론사 내부에서 병렬 처리하고, 언론사 자체는 순차 처리한다. 배치가 끝날 때마다 `results.jsonl`, `failures.jsonl`, `summary.json`, `execution.log.jsonl`을 저장한다.

In [11]:
press_listt = press_list[:1]
press_listt

['강원일보']

In [ ]:
# CURRENT_PRESS = '경향신문'  # 실행할 언론사를 1곳씩 지정

def get_press_data_version(press):
    with get_postgre_connect() as connection:
        rows = connection.execute(
            "SELECT DISTINCT data_version FROM ner_human_review "
            "WHERE review_method = 'LLM검수' AND press = %s ORDER BY data_version DESC",
            [press],
        ).fetchall()
    data_versions = [row[0] for row in rows]
    if not data_versions:
        raise ValueError(f'ner_human_review에 데이터가 없습니다: {press}')
    if DATA_VERSION and DATA_VERSION not in data_versions:
        raise ValueError(f'{press}에 DATA_VERSION이 없습니다: {DATA_VERSION}')
    if DATA_VERSION:
        return DATA_VERSION
    if len(data_versions) > 1:
        raise ValueError(f'{press}에 여러 data_version이 있습니다: {data_versions}')
    return data_versions[0]

for current_press in press_listt:
    CURRENT_PRESS = current_press

    PRESS_DATA_VERSION = get_press_data_version(CURRENT_PRESS)
    print(f'실행 언론사: {CURRENT_PRESS} | data_version: {PRESS_DATA_VERSION}')

    if RUN_API:
        summaries = run_press_list(
            [CURRENT_PRESS],
            OUTPUT_ROOT,
            data_version=PRESS_DATA_VERSION,
            model=MODEL,
            batch_size=BATCH_SIZE,
            max_workers=MAX_WORKERS,
            max_retries=MAX_RETRIES,
            backoff_seconds=BACKOFF_SECONDS,
            max_backoff_seconds=MAX_BACKOFF_SECONDS,
            request_timeout_seconds=REQUEST_TIMEOUT_SECONDS,
            max_input_chars=MAX_INPUT_CHARS,
            progress=lambda event: print(event),
        )
        display(pd.DataFrame(summaries))
    else:
        print('RUN_API=False: 실제 Decision API 호출을 실행하지 않았습니다.')

##### 4. 실행 현황과 결과 확인

In [ ]:
def load_summaries(output_root):
    paths = sorted(Path(output_root).glob('press/*/summary.json'))
    rows = []
    for path in paths:
        rows.append(__import__('json').loads(path.read_text(encoding='utf-8')).copy())
        rows[-1]['summary_path'] = str(path)
    return pd.DataFrame(rows)

summary_frame = load_summaries(OUTPUT_ROOT)
if CURRENT_PRESS:
    summary_frame = summary_frame[summary_frame['press'] == CURRENT_PRESS]
display(summary_frame)

press_dir = Path(OUTPUT_ROOT) / 'press' / CURRENT_PRESS
result_path = press_dir / 'results.jsonl'
failure_path = press_dir / 'failures.jsonl'
if result_path.exists():
    result_frame = pd.read_json(result_path, lines=True)
    display(result_frame)
if failure_path.exists():
    failure_frame = pd.read_json(failure_path, lines=True)
    display(failure_frame)
print('결과 루트:', OUTPUT_ROOT)

##### 실행 순서(언론사별 수동 실행)

1. 위 설정 셀과 1·2번 셀을 먼저 실행한다.
2. 3번 셀의 `CURRENT_PRESS`에 실행할 언론사 1곳을 지정하고 3번 셀만 실행한다.
3. 실행이 끝나면 4번 셀을 실행해 해당 언론사의 summary와 결과를 확인한다.
4. 다음 언론사를 실행할 때는 `CURRENT_PRESS`만 바꾼 뒤 3번과 4번 셀을 반복 실행한다.
5. 모든 언론사는 같은 `OUTPUT_ROOT` 아래 언론사별 폴더에 저장된다.

##### Decision API 결과 검증

In [13]:
import json
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(r'C:\work\ner')
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'decision_2' / 'run_20261120_014739'
CONFIDENCE_THRESHOLD = 0.5

def read_jsonl(path):
    rows = []
    parse_errors = []
    if not path.exists():
        return rows, ['파일 없음']
    with path.open(encoding='utf-8') as file:
        for line_no, line in enumerate(file, start=1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as error:
                parse_errors.append(f'{line_no}행: {error.msg}')
    return rows, parse_errors

def answer_by_suffix(result, suffix):
    return next((answer for answer in result.get('api', {}).get('answers', [])
                 if str(answer.get('name', '')).endswith(suffix)), {})

summary_rows = []
all_candidate_rows = []
example_result = None

for press_dir in sorted((OUTPUT_ROOT / 'press').iterdir()):
    if not press_dir.is_dir():
        continue
    press = press_dir.name
    summary_path = press_dir / 'summary.json'
    result_path = press_dir / 'results.jsonl'
    failure_path = press_dir / 'failures.jsonl'
    log_path = press_dir / 'execution.log.jsonl'

    # 1. 언론사별 4개 파일의 존재 여부와 JSON/JSONL 파싱 오류를 확인한다.
    missing_files = [str(path.name) for path in (summary_path, result_path, failure_path, log_path)
                    if not path.exists()]
    summary = json.loads(summary_path.read_text(encoding='utf-8')) if summary_path.exists() else {}
    results, result_parse_errors = read_jsonl(result_path)
    failures, failure_parse_errors = read_jsonl(failure_path)
    logs, log_parse_errors = read_jsonl(log_path)

    # 2. summary.completed, results 행 수, 완료 로그 수가 일치하는지 확인한다.
    completed_log_count = sum(log.get('event') == 'completed' for log in logs)
    count_match = (summary.get('completed') == len(results) == completed_log_count)

    # 3. summary.failed와 failures 행 수 및 실패 로그 수가 일치하는지 확인한다.
    failed_log_count = sum(log.get('event') in {'request_failed', 'parse_error'} for log in logs)
    failure_count_match = summary.get('failed') == len(failures) == failed_log_count

    # 4. 실패·미처리·대기 상태가 없는지 확인한다. output_tokens=0은 Decision API 정상값이다.
    run_complete = (summary.get('status') == 'completed' and summary.get('failed') == 0
                   and summary.get('pending') == 0 and summary.get('unprocessed') == 0)

    # 5. 결과 식별자(data_version, press, doc_id, sentence_id)의 중복을 확인한다.
    result_keys = [(row.get('data_version'), row.get('press'), row.get('doc_id'), row.get('sentence_id'))
                   for row in results]
    duplicate_count = len(result_keys) - len(set(result_keys))

    # 6. 모든 결과가 같은 실행의 schema/run/model/prompt 메타데이터인지 확인한다.
    metadata_error_count = sum(
        row.get('schema_version') != summary.get('schema_version')
        or row.get('run_id') != summary.get('run_id')
        or row.get('press') != summary.get('press')
        or row.get('model') != summary.get('model')
        or row.get('prompt_version') != summary.get('prompt_version')
        for row in results
    )

    # 7. 휴먼검수에 필요한 tagging_error/tagging_merge/source/api 구조를 확인한다.
    required_error_fields = {'is_error', 'error_type', 'confidence', 'probabilities'}
    required_merge_fields = {'needed', 'confidence', 'probabilities'}
    structure_error_count = sum(
        not required_error_fields.issubset(row.get('tagging_error', {}))
        or not required_merge_fields.issubset(row.get('tagging_merge', {}))
        or not {'existing_entities', 'active_rule_evidence'}.issubset(row.get('source', {}))
        or not {'usage', 'answers'}.issubset(row.get('api', {}))
        for row in results
    )

    # 8. API answer 선택값이 정규화된 tagging_error/tagging_merge 결과와 일치하는지 확인한다.
    decision_mismatch_count = 0
    for row in results:
        error_answer = answer_by_suffix(row, '__tagging_error')
        merge_answer = answer_by_suffix(row, '__tagging_merge')
        error_choice = error_answer.get('choice')
        merge_choice = merge_answer.get('choice')
        error = row.get('tagging_error', {})
        merge = row.get('tagging_merge', {})
        error_ok = error_choice in {'normal', 'wrong_major_tag', 'other_tagging_error'}
        error_ok = error_ok and error.get('is_error') == (error_choice != 'normal')
        error_ok = error_ok and (error_choice == 'normal' or error.get('error_type') == error_choice)
        merge_ok = isinstance(merge_choice, bool) and merge.get('needed') == merge_choice
        decision_mismatch_count += not (error_ok and merge_ok)

    # 9. 각 확률 배열이 3개 항목이며 합계가 1인지 확인한다.
    probability_error_count = 0
    for row in results:
        probability_lists = [row.get('tagging_error', {}).get('probabilities'),
                             row.get('tagging_merge', {}).get('probabilities')]
        for probabilities in probability_lists:
            values = [item.get('probability') for item in (probabilities or [])
                      if isinstance(item, dict) and isinstance(item.get('probability'), (int, float))]
            if len(values) != 3 or not 0.99 <= sum(values) <= 1.01:
                probability_error_count += 1

    # 10. 오류·병합 후보와 낮은 신뢰도 결과를 집계해 휴먼검수 가능 여부를 판단한다.
    candidate_count = 0
    low_confidence_count = 0
    for row in results:
        error = row.get('tagging_error', {})
        merge = row.get('tagging_merge', {})
        confidences = [value for value in (error.get('confidence'), merge.get('confidence'))
                      if isinstance(value, (int, float))]
        low_confidence = bool(confidences) and min(confidences) <= CONFIDENCE_THRESHOLD
        is_candidate = bool(error.get('is_error') or merge.get('needed') or low_confidence)
        candidate_count += is_candidate
        low_confidence_count += low_confidence
        if is_candidate and len(all_candidate_rows) < 100:
            all_candidate_rows.append({
                'press': row.get('press'), 'doc_id': row.get('doc_id'),
                'sentence_id': row.get('sentence_id'), 'original_form': row.get('original_form'),
                'tagging_error': error.get('error_type') or 'normal',
                'tagging_merge': merge.get('needed'),
                'error_confidence': error.get('confidence'),
                'merge_confidence': merge.get('confidence'),
            })
        if example_result is None and results:
            example_result = results[0]

    human_review_ready = bool(
        not missing_files and not result_parse_errors and not failure_parse_errors and not log_parse_errors
        and count_match and failure_count_match and run_complete and duplicate_count == 0
        and metadata_error_count == 0 and structure_error_count == 0
        and decision_mismatch_count == 0 and probability_error_count == 0
    )
    summary_rows.append({
        'press': press, 'missing_files': ', '.join(missing_files),
        'summary_completed': summary.get('completed'), 'results_rows': len(results),
        'completed_log_rows': completed_log_count, 'summary_failed': summary.get('failed'),
        'failure_rows': len(failures), 'count_match': count_match,
        'failure_count_match': failure_count_match, 'run_complete': run_complete,
        'duplicate_results': duplicate_count, 'metadata_errors': metadata_error_count,
        'structure_errors': structure_error_count, 'decision_mismatches': decision_mismatch_count,
        'probability_errors': probability_error_count, 'low_confidence': low_confidence_count,
        'human_review_candidates': candidate_count, 'human_review_ready': human_review_ready,
        'parse_errors': len(result_parse_errors) + len(failure_parse_errors) + len(log_parse_errors),
        'output_tokens': summary.get('output_tokens'),
    })

verification_frame = pd.DataFrame(summary_rows).sort_values('press')
display(verification_frame)
display(pd.DataFrame(all_candidate_rows))


,press,missing_files,summary_completed,results_rows,completed_log_rows,summary_failed,failure_rows,count_match,failure_count_match,run_complete,duplicate_results,metadata_errors,structure_errors,decision_mismatches,probability_errors,low_confidence,human_review_candidates,human_review_ready,parse_errors,output_tokens
0,강원일보,,2177,2177,2177,0,0,True,True,True,0,0,0,0,2177,1596,1599,False,0,0
1,경기일보,,2706,2706,2706,0,0,True,True,True,0,0,0,0,2706,2043,2050,False,0,0
2,경향신문,,6372,6372,6372,0,0,True,True,True,0,0,0,0,6372,4978,5006,False,0,0
3,노컷뉴스,,9076,9076,9076,0,0,True,True,True,0,0,0,0,9076,7049,7085,False,0,0
4,뉴스1,,28855,28855,28855,0,0,True,True,True,0,0,0,0,28855,22016,22118,False,0,0
5,뉴스핌,,14403,14403,14403,0,0,True,True,True,0,0,0,0,14403,10719,10775,False,0,0
6,뉴시스,,22106,22106,22106,0,0,True,True,True,0,0,0,0,22106,16732,16816,False,0,0
7,디지털타임스,,5404,5404,5404,0,0,True,True,True,0,0,0,0,5404,3893,3910,False,0,0
8,마이데일리,,7706,7706,7706,0,0,True,True,True,0,0,0,0,7706,5509,5525,False,0,0
9,부산일보,,6347,6347,6347,0,0,True,True,True,0,0,0,0,6347,4893,4901,False,0,0


,press,doc_id,sentence_id,original_form,tagging_error,tagging_merge,error_confidence,merge_confidence
0,강원일보,NLRW2600000001.10006,NLRW2600000001.10006.6.2,<2022년: DT> <20대 대선: EV>의 <강원: LC>지역 <1일차: DT>...,normal,False,0.16,1.00
1,강원일보,NLRW2600000001.10008,NLRW2600000001.10008.5.1,<6·3 대선: EV>을 앞두고 <강원도: LC>는 다시금 전국 정치의 중심에 섰다.,wrong_major_tag,False,0.18,1.00
2,강원일보,NLRW2600000001.10014,NLRW2600000001.10014.1.1,<오늘: DT>부터 사전투표… <이재명: PS>-<김문수: PS> <30일: DT>...,normal,False,0.42,1.00
3,강원일보,NLRW2600000001.10014,NLRW2600000001.10014.6.1,<29일: DT> 사전투표를 시작으로 <닷새간: DT> <6·3대선: EV> 운명을...,normal,False,0.42,1.00
4,강원일보,NLRW2600000001.10024,NLRW2600000001.10024.6.1,<2년: DT>에 한 번 전 세계를 순회하며 전시를 기획하는 유목예술팀 ‘<_비엔날...,normal,False,0.18,1.00
...,...,...,...,...,...,...,...,...
95,강원일보,NLRW2600000001.11263,NLRW2600000001.11263.5.1,이러한 상황속에 <춘천시: OG>는 <지난해: DT>부터 이미 통합 대응 시나리오를...,normal,False,0.35,1.00
96,강원일보,NLRW2600000001.11290,NLRW2600000001.11290.4.1,<인제군: OG>이 <강원특별자치도: OG> 주관 ‘<2025년: DT> <봄철: ...,wrong_major_tag,False,0.16,0.98
97,강원일보,NLRW2600000001.11296,NLRW2600000001.11296.2.1,<정부: OG>와 <더불어민주당: OG>은 <18일: DT> <올해: DT> 추가경...,normal,False,0.38,1.00
98,강원일보,NLRW2600000001.113,NLRW2600000001.113.10.1,<다음달: DT>부터 <강원특별자치도: OG> 시·군 중 처음으로 <홍천: LC>에...,normal,False,0.35,1.00


In [ ]:
# 19개 언론사 모두 실행 완료 상태
# results.jsonl, 완료 로그, summary.completed 건수가 일치
# 실패·미처리·대기 건수는 모두 0
# 중복 결과나 파싱 오류 없음

##### results.jsonl sample 1

In [15]:
# 실제 results.jsonl 예시: 강원일보 정상/태그오류 + 다른 언론사 병합오류
import json
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(r'C:\work\ner')
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'decision_2' / 'run_20261120_014739'

def first_result(press_dir, predicate):
    result_path = press_dir / 'results.jsonl'
    with result_path.open(encoding='utf-8') as file:
        for line in file:
            if not line.strip():
                continue
            result = json.loads(line)
            if predicate(result):
                return result
    return None

gangwon_dir = OUTPUT_ROOT / 'press' / '강원일보'
example_results = [
    ('정상', first_result(gangwon_dir, lambda row: not row['tagging_error']['is_error'] and not row['tagging_merge']['needed'])),
    ('태그오류', first_result(gangwon_dir, lambda row: row['tagging_error']['is_error'])),
]

merge_example = None
for press_dir in sorted((OUTPUT_ROOT / 'press').iterdir()):
    if press_dir.name == '강원일보' or not press_dir.is_dir():
        continue
    merge_example = first_result(press_dir, lambda row: row['tagging_merge']['needed'])
    if merge_example is not None:
        break
example_results.append(('병합오류', merge_example))

def load_db_example(row):
    if row is None:
        return {}, []
    review_row = {}
    rule_rows = []
    try:
        from src.config import get_postgre_connect
        with get_postgre_connect() as connection:
            key = [row['data_version'], row['press'], row['doc_id'], row['sentence_id']]
            review_columns = ['data_version', 'press', 'doc_id', 'sentence_id', 'original_form',
                              'modified_form', 'review_method', 'review_status', 'is_modified',
                              'assignee', 'updated_at']
            review_row = connection.execute(
                '''SELECT data_version, press, doc_id, sentence_id, original_form, modified_form,
                          review_method, review_status, is_modified, assignee, updated_at
                   FROM ner_human_review
                   WHERE data_version = %s AND press = %s AND doc_id = %s AND sentence_id = %s''',
                key,
            ).fetchone() or {}
            if review_row and not hasattr(review_row, 'get'):
                review_row = dict(zip(review_columns, review_row))
            rule_columns = ['entity_id', 'form', 'entity_tagging', 'entity_name', 'entity_type']
            rule_rows = connection.execute(
                '''SELECT entity_id, form, entity_tagging, entity_name, entity_type
                   FROM ner_rule_results_3
                   WHERE data_version = %s AND doc_id = %s AND sentence_id = %s
                   ORDER BY entity_id''',
                [row['data_version'], row['doc_id'], row['sentence_id']],
            ).fetchall()
            rule_rows = [rule_row if hasattr(rule_row, 'get') else dict(zip(rule_columns, rule_row))
                         for rule_row in rule_rows]
    except Exception as error:
        review_row = {'DB 조회': f'실패: {error}'}
    return review_row, rule_rows

example_rows = []
for example_type, row in example_results:
    if row is None:
        continue
    review_row, rule_rows = load_db_example(row)
    example_rows.append({
        '예시 유형': example_type,
        '언론사': row['press'],
        '식별자': f"{row['doc_id']} / {row['sentence_id']}",
        '원문': row['original_form'],
        'tagging_error': {
            'is_error': row['tagging_error']['is_error'],
            'error_type': row['tagging_error']['error_type'],
            'confidence': row['tagging_error']['confidence'],
        },
        'tagging_merge': {
            'needed': row['tagging_merge']['needed'],
            'confidence': row['tagging_merge']['confidence'],
        },
        'ner_human_review': {key: review_row.get(key) for key in
                            ('review_method', 'review_status', 'is_modified', 'assignee')},
        'ner_rule_results_3': [
            {key: rule_row.get(key) for key in
             ('entity_id', 'form', 'entity_tagging', 'entity_name', 'entity_type')}
            for rule_row in rule_rows
        ],
        '휴먼 확인': '정상은 통과, 태그오류는 태그/범위, 병합오류는 엔티티 병합 필요성을 원문과 대조',
    })

example_frame = pd.DataFrame(example_rows)
display(example_frame)
print('정상/태그오류: 강원일보, 병합오류: 실제 병합 후보가 존재하는 다른 언론사')


,예시 유형,언론사,식별자,원문,tagging_error,tagging_merge,ner_human_review,ner_rule_results_3,휴먼 확인
0,정상,강원일보,NLRW2600000001.10006 / NLRW2600000001.10006.6.2,<2022년: DT> <20대 대선: EV>의 <강원: LC>지역 <1일차: DT>...,"{'is_error': False, 'error_type': None, 'confi...","{'needed': False, 'confidence': 1.0}","{'review_method': 'LLM검수', 'review_status': '완...","[{'entity_id': 'NLRW2600000001.10006.6.2.E2', ...","정상은 통과, 태그오류는 태그/범위, 병합오류는 엔티티 병합 필요성을 원문과 대조"
1,태그오류,강원일보,NLRW2600000001.10008 / NLRW2600000001.10008.5.1,<6·3 대선: EV>을 앞두고 <강원도: LC>는 다시금 전국 정치의 중심에 섰다.,"{'is_error': True, 'error_type': 'wrong_major_...","{'needed': False, 'confidence': 1.0}","{'review_method': 'LLM검수', 'review_status': '완...","[{'entity_id': 'NLRW2600000001.10008.5.1.E1', ...","정상은 통과, 태그오류는 태그/범위, 병합오류는 엔티티 병합 필요성을 원문과 대조"
2,병합오류,경기일보,NLRW2600000002.26731 / NLRW2600000002.26731.4.1,<도: OG>는 <27일부터 12월 19일까지: DT> ‘안전전세 프로젝트’ 동참 ...,"{'is_error': False, 'error_type': None, 'confi...","{'needed': True, 'confidence': 0.38}","{'review_method': 'LLM검수', 'review_status': '미...","[{'entity_id': 'NLRW2600000002.26731.4.1.E3', ...","정상은 통과, 태그오류는 태그/범위, 병합오류는 엔티티 병합 필요성을 원문과 대조"


정상/태그오류: 강원일보, 병합오류: 실제 병합 후보가 존재하는 다른 언론사


In [28]:
example_frame['원문'][2]

'<도: OG>는 <27일부터 12월 19일까지: DT> ‘안전전세 프로젝트’ 동참 중개사무소 1천 곳과 불법 중개행위 의심 중개사무소 300곳을 대상으로 ‘<2025년: DT> <하반기: DT> 공인중개사 민·관 합동 점검’을 한다고 밝혔다.'

In [29]:
example_frame['tagging_error'][2]

{'is_error': False, 'error_type': None, 'confidence': 0.35}

In [30]:
example_frame['tagging_merge'][2]

{'needed': True, 'confidence': 0.38}

In [31]:
example_frame['ner_rule_results_3'][2]

[{'entity_id': 'NLRW2600000002.26731.4.1.E3',
  'form': '<도: OG>는 <27일부터 12월 19일까지: DT> ‘안전전세 프로젝트’ 동참 중개사무소 1천 곳과 불법 중개행위 의심 중개사무소 300곳을 대상으로 ‘<2025년: DT> <하반기: DT> 공인중개사 민·관 합동 점검’을 한다고 밝혔다.',
  'entity_tagging': '<2025년: DT>',
  'entity_name': '2025년',
  'entity_type': 'DT'},
 {'entity_id': 'NLRW2600000002.26731.4.1.E4',
  'form': '<도: OG>는 <27일부터 12월 19일까지: DT> ‘안전전세 프로젝트’ 동참 중개사무소 1천 곳과 불법 중개행위 의심 중개사무소 300곳을 대상으로 ‘<2025년: DT> <하반기: DT> 공인중개사 민·관 합동 점검’을 한다고 밝혔다.',
  'entity_tagging': '<하반기: DT>',
  'entity_name': '하반기',
  'entity_type': 'DT'}]

##### Decision 결과 정제

In [36]:
# Decision 결과 정제 저장: 언론사 1곳씩 실행
import json
from collections import Counter
from pathlib import Path

PROJECT_ROOT = Path(r'C:\work\ner')
DECISION_ROOT = PROJECT_ROOT / 'outputs' / 'decision_2' / 'run_20261120_014739' / 'press'
CLEANED_ROOT = DECISION_ROOT
CONFIDENCE_THRESHOLD = 0.8

def make_cleaned_row(result):
    error = result.get('tagging_error') or {}
    merge = result.get('tagging_merge') or {}
    source = result.get('source') or {}
    api = result.get('api') or {}
    error_is_error = bool(error.get('is_error'))
    merge_needed = bool(merge.get('needed'))
    confidences = [value for value in (error.get('confidence'), merge.get('confidence')) if isinstance(value, (int, float))]
    min_confidence = min(confidences) if confidences else None
    low_confidence = min_confidence is not None and min_confidence <= CONFIDENCE_THRESHOLD

    # 단계 2: 오류 유형을 정하고, 낮은 신뢰도 정상건도 검수 후보로 표시한다.
    if error_is_error:
        decision_type = 'tagging_error'
    elif merge_needed:
        decision_type = 'tagging_merge'
    else:
        decision_type = 'normal'
    review_reason = []
    if error_is_error:
        review_reason.append('tagging_error')
    if merge_needed:
        review_reason.append('tagging_merge')
    if low_confidence:
        review_reason.append('low_confidence')

    # 단계 3: 현재 화면의 식별키와 연결할 원문/엔티티/규칙 근거를 보존한다.
    rule_ids = sorted(set((error.get('candidate_rule_ids') or []) + (merge.get('candidate_rule_ids') or [])))
    return {
        'run_id': result.get('run_id'),
        'schema_version': result.get('schema_version'),
        'data_version': result.get('data_version'),
        'press': result.get('press'),
        'doc_id': result.get('doc_id'),
        'sentence_id': result.get('sentence_id'),
        'executed_at': result.get('executed_at'),
        'model': result.get('model'),
        'prompt_version': result.get('prompt_version'),
        'status': result.get('status'),
        'original_form': result.get('original_form'),
        'decision_type': decision_type,
        'review_candidate': bool(error_is_error or merge_needed or low_confidence),
        'low_confidence': low_confidence,
        'min_confidence': min_confidence,
        'tagging_error_is_error': error_is_error,
        'tagging_error_type': error.get('error_type'),
        'tagging_error_confidence': error.get('confidence'),
        'tagging_error_probabilities': error.get('probabilities'),
        'tagging_merge_needed': merge_needed,
        'tagging_merge_confidence': merge.get('confidence'),
        'tagging_merge_probabilities': merge.get('probabilities'),
        'review_reason': review_reason,
        'candidate_rule_ids': rule_ids,
        'existing_entities': source.get('existing_entities', []),
        'active_rule_evidence': source.get('active_rule_evidence', []),
        'api_usage': api.get('usage') or {},
        'api_answers': api.get('answers') or [],
    }

def clean_one_press(press):
    # 단계 4: 결과를 메모리에 모두 올리지 않고 JSONL을 한 줄씩 읽어 저장한다.
    source_path = DECISION_ROOT / press / 'results.jsonl'
    output_dir = CLEANED_ROOT / press
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / 'results_cleaned.jsonl'
    counts = Counter()
    total = 0
    with source_path.open(encoding='utf-8') as source, output_path.open('w', encoding='utf-8') as target:
        for line in source:
            if not line.strip():
                continue
            cleaned = make_cleaned_row(json.loads(line))
            target.write(json.dumps(cleaned, ensure_ascii=False, separators=(',', ':')) + '\n')
            counts[cleaned['decision_type']] += 1
            counts['review_candidate'] += int(cleaned['review_candidate'])
            counts['low_confidence'] += int(cleaned['low_confidence'])
            total += 1

    # 단계 5: 언론사별 적재 건수와 검수 후보 수를 별도 요약 파일로 저장한다.
    summary = {
        'press': press,
        'source_path': str(source_path),
        'output_path': str(output_path),
        'total': total,
        'normal': counts['normal'],
        'tagging_error': counts['tagging_error'],
        'tagging_merge': counts['tagging_merge'],
        'review_candidate': counts['review_candidate'],
        'low_confidence': counts['low_confidence'],
    }
    (output_dir / 'results_cleaned_summary.json').write_text(
        json.dumps(summary, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    return summary


In [37]:
press_list = ['강원일보', '경기일보', '경향신문', '노컷뉴스', '뉴스1', '뉴스핌', '뉴시스', '디지털타임스', '마이데일리', '부산일보', '서울신문', '세계일보', '스포츠경향', '스포츠투데이', '아시아투데이', '이데일리', '이투데이', '일간투데이', '충청일보']
len(press_list)

19

In [38]:
# 단계 1: 실행할 언론사를 하나만 지정한다. 다음 언론사는 이 값만 바꿔 다시 실행한다.
# PRESS_TO_PROCESS = '강원일보'

for PRESS_TO_PROCESS in press_list:
    print(clean_one_press(PRESS_TO_PROCESS))

{'press': '강원일보', 'source_path': 'C:\\work\\ner\\outputs\\decision_2\\run_20261120_014739\\press\\강원일보\\results.jsonl', 'output_path': 'C:\\work\\ner\\outputs\\decision_2\\run_20261120_014739\\press\\강원일보\\results_cleaned.jsonl', 'total': 2177, 'normal': 1919, 'tagging_error': 258, 'tagging_merge': 0, 'review_candidate': 2116, 'low_confidence': 2116}
{'press': '경기일보', 'source_path': 'C:\\work\\ner\\outputs\\decision_2\\run_20261120_014739\\press\\경기일보\\results.jsonl', 'output_path': 'C:\\work\\ner\\outputs\\decision_2\\run_20261120_014739\\press\\경기일보\\results_cleaned.jsonl', 'total': 2706, 'normal': 2378, 'tagging_error': 321, 'tagging_merge': 7, 'review_candidate': 2629, 'low_confidence': 2629}
{'press': '경향신문', 'source_path': 'C:\\work\\ner\\outputs\\decision_2\\run_20261120_014739\\press\\경향신문\\results.jsonl', 'output_path': 'C:\\work\\ner\\outputs\\decision_2\\run_20261120_014739\\press\\경향신문\\results_cleaned.jsonl', 'total': 6372, 'normal': 5391, 'tagging_error': 974, 'tagging_me

##### ner_llm_decision 결과 검증

In [ ]:
# 화면에 사용할 검수 대상 테이블 확인용 조회입니다.
# ner_human_review를 기준으로 하므로 Decision 결과가 없는 문장도 함께 확인할 수 있습니다.
import pandas as pd
from src.config import get_postgre_connect

TARGET_RUN_ID = 'run_20261120_014739'  # 확인할 Decision 실행 ID
TARGET_DATA_VERSION = None              # 예: '20261117_192958_강원일보'
TARGET_PRESS = None                     # 예: '강원일보'
DISPLAY_LIMIT = 100

query = """
    SELECT
        h.data_version,
        h.press,
        h.doc_id,
        h.sentence_id,
        h.original_form,
        h.modified_form,
        h.review_method,
        h.review_status,
        d.run_id AS decision_run_id,
        CASE
            WHEN d.run_id IS NULL THEN '결과없음'
            WHEN d.tagging_error_is_error THEN '태그 오류'
            WHEN d.tagging_merge_needed THEN '태깅 병합'
            WHEN d.low_confidence THEN '저신뢰도'
            ELSE '정상'
        END AS review_target,
        d.status AS decision_status,
        d.decision_type,
        d.review_candidate,
        d.low_confidence,
        d.min_confidence,
        d.tagging_error_type,
        d.tagging_error_confidence,
        d.tagging_merge_confidence,
        d.review_reason,
        d.candidate_rule_ids,
        d.existing_entities,
        d.active_rule_evidence,
        d.api_answers
    FROM ner_human_review AS h
    LEFT JOIN ner_llm_decision AS d
      ON d.data_version = h.data_version
     AND d.press = h.press
     AND d.doc_id = h.doc_id
     AND d.sentence_id = h.sentence_id
     AND d.run_id = %s
    WHERE h.review_method = 'LLM검수'
"""
params = [TARGET_RUN_ID]
if TARGET_DATA_VERSION:
    query += ' AND h.data_version = %s'
    params.append(TARGET_DATA_VERSION)
if TARGET_PRESS:
    query += ' AND h.press = %s'
    params.append(TARGET_PRESS)
query += ' ORDER BY h.press, h.data_version, h.doc_id, h.sentence_id LIMIT %s'
params.append(DISPLAY_LIMIT)

with get_postgre_connect() as connection:
    with connection.cursor() as cursor:
        cursor.execute(query, params)
        columns = [column[0] for column in cursor.description]
        rows = cursor.fetchall()

review_table = pd.DataFrame(rows, columns=columns)

##### 결과 재가공
- 기존 run_llm_decision 삭제
- C:\work\ner\outputs\decision_2\run_20261120_014739\press 
- 위 경로 저장된 results_cleaned.jsonl 파일 기준 재가공 후 ner_decision_results 테이블 사용

- 의심 수준 분류 기준
  - 태그오류 확률: wrong_major_tag의 확률
  - 태깅병합 확률: True의 확률
  - 신뢰도: 낮은 신뢰도의 정상 판단을 추가 확인하기 위한 보조 지표
  - 임계값: 휴먼 검수 결과를 기반으로 추후 조정할 초기 설정값
  - 오류 확률을 우선 활용하고, 신뢰도를 보조 지표로 활용

In [ ]:
import pandas as pd
import numpy as np

jsonl_path = r"C:\work\ner\outputs\decision_2\run_20261120_014739\press\강원일보\results_cleaned.jsonl"
df = pd.read_json(jsonl_path, lines=True)

# 검출 개체명 및 룰 정보 가공
df["detected_entities"] = df["existing_entities"].apply(
    lambda x: ", ".join(e["target_span"] for e in x)
)
df["candidate_rule_ids"] = df["candidate_rule_ids"].apply(
    lambda x: ", ".join(x)
)

# 오류 확률 추출
def get_probability(probabilities, value):
    return next(
        (p["probability"] for p in probabilities if p["value"] == value),
        np.nan
    ) if isinstance(probabilities, list) else np.nan

df["tagging_error_probability"] = df["tagging_error_probabilities"].apply(
    lambda x: get_probability(x, "wrong_major_tag")
)

df["tagging_merge_probability"] = df["tagging_merge_probabilities"].apply(
    lambda x: get_probability(x, True)
)

# 결측값 검증
assert df[
    [
        "tagging_error_probability",
        "tagging_error_confidence",
        "tagging_merge_probability",
        "tagging_merge_confidence",
    ]
].notna().all().all(), "Decision 결과 결측값 존재"

# 태그오류 의심 수준
df["tagging_error_result"] = np.select(
    [
        (df.tagging_error_type == "wrong_major_tag")
        | (df.tagging_error_probability >= 0.4),

        (df.tagging_error_probability >= 0.2)
        | (df.tagging_error_confidence <= 0.2),
    ],
    ["강한 의심", "중간 의심"],
    default="약한 의심"
)

# 태깅병합 의심 수준
df["tagging_merge_result"] = np.select(
    [
        (df.tagging_merge_needed == True)
        | (df.tagging_merge_probability >= 0.1),

        (df.tagging_merge_probability >= 0.01)
        | (df.tagging_merge_confidence <= 0.95),
    ],
    ["강한 의심", "중간 의심"],
    default="약한 의심"
)

# 최종 컬럼 선택
columns = [
    # 매핑 키
    "run_id",
    "data_version",
    "press",
    "doc_id",
    "sentence_id",

    # 검출 정보
    "detected_entities",
    "candidate_rule_ids",

    # 태그오류 판단
    "tagging_error_probability",
    "tagging_error_confidence",
    "tagging_error_result",

    # 태깅병합 판단
    "tagging_merge_probability",
    "tagging_merge_confidence",
    "tagging_merge_result",

    # Response 대상
    "review_candidate",
]

decision_df = df[columns].copy()
decision_df

,run_id,data_version,press,doc_id,sentence_id,detected_entities,candidate_rule_ids,tagging_error_probability,tagging_error_confidence,tagging_error_result,tagging_merge_probability,tagging_merge_confidence,tagging_merge_result,review_candidate
0,run_20261120_014739,20261117_192958_강원일보,강원일보,NLRW2600000001.10006,NLRW2600000001.10006.6.2,<20대 대선: EV>,EXT_E005_EV_004,0.30,0.16,중간 의심,0.00,1.00,약한 의심,True
1,run_20261120_014739,20261117_192958_강원일보,강원일보,NLRW2600000001.10008,NLRW2600000001.10008.5.1,<6·3 대선: EV>,EXT_E005_EV_004,0.45,0.18,강한 의심,0.00,1.00,약한 의심,True
2,run_20261120_014739,20261117_192958_강원일보,강원일보,NLRW2600000001.10014,NLRW2600000001.10014.1.1,<오늘: DT>,EXT_E004_DT_004,0.13,0.42,약한 의심,0.00,1.00,약한 의심,True
3,run_20261120_014739,20261117_192958_강원일보,강원일보,NLRW2600000001.10014,NLRW2600000001.10014.6.1,<6·3대선: EV>,EXT_E005_EV_004,0.29,0.42,중간 의심,0.00,1.00,약한 의심,True
4,run_20261120_014739,20261117_192958_강원일보,강원일보,NLRW2600000001.1002,NLRW2600000001.1002.4.1,<3월: DT>,EXT_E004_DT_004,0.05,0.76,약한 의심,0.00,1.00,약한 의심,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2172,run_20261120_014739,20261117_192958_강원일보,강원일보,NLRW2600000001.9993,NLRW2600000001.9993.2.1,<2015년: DT>,EXT_E004_DT_004,0.20,0.25,중간 의심,0.00,1.00,약한 의심,True
2173,run_20261120_014739,20261117_192958_강원일보,강원일보,NLRW2600000001.9993,NLRW2600000001.9993.5.1,<2015년: DT>,EXT_E004_DT_004,0.22,0.22,중간 의심,0.00,1.00,약한 의심,True
2174,run_20261120_014739,20261117_192958_강원일보,강원일보,NLRW2600000001.9993,NLRW2600000001.9993.7.1,"<2015년: DT>, <지난해: DT>",EXT_E004_DT_004,0.09,0.20,중간 의심,0.46,0.08,강한 의심,True
2175,run_20261120_014739,20261117_192958_강원일보,강원일보,NLRW2600000001.9995,NLRW2600000001.9995.4.1,<강원특별자치도: OG>,EXT_S003,0.30,0.30,중간 의심,0.01,0.98,중간 의심,True


In [105]:
decision_df[["tagging_error_result", "tagging_merge_result"]]\
.apply(pd.Series.value_counts).reindex(["강한 의심", "중간 의심", "약한 의심"]).fillna(0).astype(int)

,tagging_error_result,tagging_merge_result
강한 의심,133,65
중간 의심,611,316
약한 의심,1433,1796


In [ ]:
decision_df[decision_df['tagging_error_result'] == '약한 의심']

,run_id,data_version,press,doc_id,sentence_id,detected_entities,candidate_rule_ids,tagging_error_probability,tagging_error_confidence,tagging_error_result,tagging_merge_probability,tagging_merge_confidence,tagging_merge_result,review_candidate
2,run_20261120_014739,20261117_192958_강원일보,강원일보,NLRW2600000001.10014,NLRW2600000001.10014.1.1,<오늘: DT>,EXT_E004_DT_004,0.13,0.42,약한 의심,0.00,1.00,약한 의심,True
4,run_20261120_014739,20261117_192958_강원일보,강원일보,NLRW2600000001.1002,NLRW2600000001.1002.4.1,<3월: DT>,EXT_E004_DT_004,0.05,0.76,약한 의심,0.00,1.00,약한 의심,True
6,run_20261120_014739,20261117_192958_강원일보,강원일보,NLRW2600000001.10031,NLRW2600000001.10031.2.1,<22대 국회의원 선거: EV>,EXT_E005_EV_004,0.10,0.74,약한 의심,0.00,1.00,약한 의심,True
8,run_20261120_014739,20261117_192958_강원일보,강원일보,NLRW2600000001.1004,NLRW2600000001.1004.7.1,<올해 상반기: DT>,EXT_E004_DT_005,0.08,0.61,약한 의심,0.00,1.00,약한 의심,True
9,run_20261120_014739,20261117_192958_강원일보,강원일보,NLRW2600000001.1004,NLRW2600000001.1004.9.1,<연말: DT>,EXT_E004_DT_004,0.04,0.58,약한 의심,0.00,1.00,약한 의심,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2166,run_20261120_014739,20261117_192958_강원일보,강원일보,NLRW2600000001.9962,NLRW2600000001.9962.7.1,<3개월: DT>,EXT_E004_DT_001,0.07,0.30,약한 의심,0.00,1.00,약한 의심,True
2168,run_20261120_014739,20261117_192958_강원일보,강원일보,NLRW2600000001.9975,NLRW2600000001.9975.5.1,<다음 달: DT>,EXT_E004_DT_005,0.10,0.35,약한 의심,0.01,0.98,중간 의심,True
2169,run_20261120_014739,20261117_192958_강원일보,강원일보,NLRW2600000001.9981,NLRW2600000001.9981.4.2,<이날: DT>,EXT_E004_DT_001,0.11,0.34,약한 의심,0.00,1.00,약한 의심,True
2170,run_20261120_014739,20261117_192958_강원일보,강원일보,NLRW2600000001.9987,NLRW2600000001.9987.10.1,<올해: DT>,EXT_S003,0.06,0.69,약한 의심,0.00,1.00,약한 의심,True


- 실제 재가공 및 적재는 run_dbinsert의 3-2에서 실행